In [ ]:
# Cell 1: Imports, Load Clustered Dataset, Constants & Helpers
import inspect
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules

df = pd.read_csv('data/processed/psu_financial_literacy_clustered.csv')

PERSONA_COLS = [
    'Persona: Active Transactors (C0)',
    'Persona: At-Risk Novices (C1)',
    'Persona: Disciplined Savers (C2)'
]
OECD_COLS = [
    'OECD: Meets Target (>=70)',
    'OECD: Below Target (<70)'
]
DISPLAY_COLS = ['Antecedents (IF)', 'Consequents (THEN)', 'support', 'confidence', 'lift']

def get_rules(frequent_itemsets, metric='lift', min_threshold=1.15):
    """Handles signature differences across mlxtend versions and empty itemsets."""
    if frequent_itemsets.empty:
        return pd.DataFrame()
    sig = inspect.signature(association_rules)
    if 'num_itemsets' in sig.parameters:
        return association_rules(
            frequent_itemsets,
            num_itemsets=len(frequent_itemsets),
            metric=metric,
            min_threshold=min_threshold
        )
    return association_rules(
        frequent_itemsets,
        metric=metric,
        min_threshold=min_threshold
    )

def format_rules(rules_df, sort_by=['lift', 'confidence'], ascending=False):
    """Adds readable IF/THEN columns and sorts rules cleanly."""
    if rules_df.empty:
        return pd.DataFrame(columns=DISPLAY_COLS)
    out = rules_df.copy()
    out['Antecedents (IF)'] = out['antecedents'].apply(lambda s: ', '.join(sorted(s)))
    out['Consequents (THEN)'] = out['consequents'].apply(lambda s: ', '.join(sorted(s)))
    return out.sort_values(by=sort_by, ascending=ascending)

print(f"Loaded dataset: {df.shape[0]} respondents, {df.shape[1]} columns")
print(f"OECD Below-Target count: {(df['dfl_class'] == 0).sum()} ({(df['dfl_class'] == 0).mean()*100:.1f}%)")

In [ ]:
# Cell 2: Construct Curated Boolean Transaction Basket for Apriori
basket_data = {
    # 1. Core Financial Habits (>=4 = Strong Habit, <=2 = Weak Habit)
    'Habit: Budgets Regularly': df['q26_prepare_budget'] >= 4,
    'Habit: Monitors Expenses': df['q27_monitor_expenses'] >= 4,
    'Habit: Saves Before Spending': df['q28_save_before_spending'] >= 4,
    'Habit: Avoids Impulse Buys': df['q31_avoid_unnecessary_purchases'] >= 4,
    'Habit: Reviews Tx History': df['q32_review_tx_history'] >= 4,
    'Habit: Saves for Emergencies': df['q40_save_for_emergencies'] >= 4,
    'Habit: Needs-Driven Spending': df['q42_decisions_based_on_needs'] >= 4,

    # Low confidence (mean <= 3.25 on core perceived knowledge) & weak-habit indicators (<= 2)
    'Low Perceived DFS Knowledge': (
        df[['q11_understand_how_dfs_work', 'q12_know_check_tx_and_balance',
            'q13_understand_dfs_fees', 'q14_know_protect_info']].mean(axis=1) <= 3.25
    ),
    'Weak Saving Habit': df['q28_save_before_spending'] <= 2,
    'Weak Budgeting Habit': df['q26_prepare_budget'] <= 2,

    # 2. Digital Financial Service Usage & Credit Attitudes
    'Usage: Frequent DFS (>=Often)': df['q08_dfs_usage_frequency'] >= 4,
    'Usage: Mobile Banking': df['q07_mobile_banking'] == 1,
    'Usage: Online Payments': df['q07_online_payment'] == 1,
    'Usage: Digital Savings/Inv': df['q07_digital_savings_inv'] == 1,
    'Usage: BNPL Platform': df['q07_bnpl'] == 1,
    'Purpose: Pay Bills': df['q09_pay_bills'] == 1,
    'Purpose: Save Money': df['q09_save_money'] == 1,
    'Purpose: Transfer Accounts': df['q09_transfer_accounts'] == 1,
    'Purpose: Check Balance': df['q09_check_balance'] == 1,
    'Credit: Open to BNPL (>=Likely)': df['q43_likelihood_use_bnpl'] >= 4,
    'Credit: Open to Loans (>=Likely)': df['q44_likelihood_use_loan'] >= 4,
    'Experienced Scam Attempt': df['q49_scam_yes'] == 1,

    # 3. Spending Triggers (q45 & q46)
    'First Action: Save': df['q45_save'] == 1,
    'First Action: Essentials': df['q45_essentials'] == 1,
    'First Action: Divide Planned': df['q45_divide_planned'] == 1,
    'Trigger: Friends/Social': df['q46_friends_social'] == 1,
    'Trigger: Online Ads/Social Media': df['q46_online_ads'] == 1,
    'Trigger: Discounts & Promos': df['q46_discounts_promos'] == 1,
    'Trigger: Digital Convenience': df['q46_digital_convenience'] == 1,

    # 4. Financial Challenges (q47) & Improvement Priorities (q50)
    'Challenge: Limited Allowance': df['q47_limited_allowance'] == 1,
    'Challenge: Difficulty Saving': df['q47_diff_saving'] == 1,
    'Challenge: Difficulty Budgeting': df['q47_diff_budgeting'] == 1,
    'Challenge: Unexpected Expenses': df['q47_unexpected_expenses'] == 1,
    'Challenge: Impulse Buying': df['q47_impulse_buying'] == 1,
    'Challenge: Scam Concern': df['q47_scam_concern'] == 1,
    'Improve: Budgeting': df['q50_budgeting'] == 1,
    'Improve: Saving': df['q50_saving'] == 1,
    'Improve: Recognizing Scams': df['q50_recognizing_scams'] == 1,
    'Improve: Future Planning': df['q50_future_planning'] == 1,

    # 5. Outcomes: K-Means Personas (Notebook 03) & OECD 70/100 Status
    'Persona: Active Transactors (C0)': df['kmeans_cluster'] == 0,
    'Persona: At-Risk Novices (C1)': df['kmeans_cluster'] == 1,
    'Persona: Disciplined Savers (C2)': df['kmeans_cluster'] == 2,
    'OECD: Meets Target (>=70)': df['dfl_class'] == 1,
    'OECD: Below Target (<70)': df['dfl_class'] == 0
}

basket = pd.DataFrame(basket_data, index=df.index).astype(bool)

print(f"Transaction basket shape: {basket.shape[0]} students x {basket.shape[1]} behavioral items")
print("\nTop 10 Most Frequent Items in Basket:")
print((basket.mean().sort_values(ascending=False).head(10) * 100).round(1).astype(str) + '%')

In [ ]:
# Cell 3: Cohort-Wide Behavioral Rules (Pure Habits, Usage, Triggers & Challenges)
behavior_basket = basket.drop(columns=PERSONA_COLS + OECD_COLS)

freq_behavior = apriori(behavior_basket, min_support=0.20, use_colnames=True, max_len=3)
rules_behavior = get_rules(freq_behavior, metric='lift', min_threshold=1.20)
rules_behavior = rules_behavior[rules_behavior['confidence'] >= 0.65]
rules_behavior = format_rules(rules_behavior)

print(f"Total cohort-wide behavioral rules found: {len(rules_behavior)}")
print("\n=== Top 15 Strongest Behavioral Association Rules (by Lift) ===")
print(rules_behavior[DISPLAY_COLS].head(15).round(3).to_string(index=False))

In [ ]:
# Cell 4: Targeted Rules Explaining All 3 K-Means Student Personas
persona_basket = basket.drop(columns=OECD_COLS)

freq_persona = apriori(persona_basket, min_support=0.06, use_colnames=True, max_len=3)
rules_persona = get_rules(freq_persona, metric='lift', min_threshold=1.25)

if not rules_persona.empty:
    rules_persona = rules_persona[
        rules_persona['consequents'].apply(lambda x: len(x) == 1 and next(iter(x)) in PERSONA_COLS)
        & (rules_persona['confidence'] >= 0.55)
    ]

rules_persona = format_rules(
    rules_persona,
    sort_by=['Consequents (THEN)', 'lift', 'confidence'],
    ascending=[True, False, False]
)

for persona in PERSONA_COLS:
    subset = rules_persona[rules_persona['Consequents (THEN)'] == persona]
    print(f"\n=== Top Behavioral Rules Predicting: {persona} ({len(subset)} rules) ===")
    if not subset.empty:
        print(subset[DISPLAY_COLS].head(5).round(3).to_string(index=False))
    else:
        print("No rules met the current support/confidence threshold.")

In [ ]:
# Cell 5: Leakage-Free Risk Rules for Below-OECD-70 Students
score_derived_cols = [
    col for col in basket.columns
    if col.startswith('Habit:') or col.startswith('Weak ') or col == 'Low Perceived DFS Knowledge'
]
oecd_basket_clean = basket.drop(columns=PERSONA_COLS + score_derived_cols)

freq_oecd = apriori(oecd_basket_clean, min_support=0.04, use_colnames=True, max_len=3)
rules_oecd = get_rules(freq_oecd, metric='lift', min_threshold=1.30)

if not rules_oecd.empty:
    risk_rules = rules_oecd[
        rules_oecd['consequents'].apply(lambda x: x == frozenset({'OECD: Below Target (<70)'}))
        & (rules_oecd['confidence'] >= 0.20)
    ]
else:
    risk_rules = pd.DataFrame()

risk_rules = format_rules(risk_rules)

print(f"Total Leakage-Free At-Risk (<70 OECD) rules found: {len(risk_rules)}")
print("\n=== Top 10 External Risk Patterns Associated with Below-Target DFL (<70) ===")
if not risk_rules.empty:
    print(risk_rules[DISPLAY_COLS].head(10).round(3).to_string(index=False))
else:
    print("No leakage-free at-risk rules met the threshold (motivates multivariate ML in Notebook 05).")

In [ ]:
# Cell 6: Visualize Behavioral Rules & Export Non-Empty Summary Tables
if not rules_behavior.empty:
    plt.figure(figsize=(9, 5.5))
    scatter = plt.scatter(
        rules_behavior['support'],
        rules_behavior['confidence'],
        c=rules_behavior['lift'],
        cmap='viridis',
        alpha=0.8,
        s=70,
        edgecolors='black',
        linewidths=0.4
    )
    cbar = plt.colorbar(scatter)
    cbar.set_label('Lift (Association Strength)')
    plt.title('Apriori Behavioral Association Rules: Support vs. Confidence (n = 118)')
    plt.xlabel('Support (Share of Cohort)')
    plt.ylabel('Confidence (Conditional Probability)')
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

exports = [
    (rules_behavior[DISPLAY_COLS].head(30), 'data/processed/apriori_behavioral_rules.csv'),
    (rules_persona[DISPLAY_COLS], 'data/processed/apriori_persona_rules.csv'),
    (risk_rules[DISPLAY_COLS].head(20), 'data/processed/apriori_at_risk_rules.csv')
]

for table, path in exports:
    if not table.empty:
        table.round(4).to_csv(path, index=False)
        print(f"Exported {len(table)} rules -> {path}")
    else:
        print(f"Skipped empty export: {path}")